In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
ROOT = Path("..")

SUB_SES_SPLITS = {
    "train": [
        ("subj01", (0, 35)),
        ("subj02", (0, 35)),
        ("subj03", (0, 27)),
        ("subj06", (0, 27)),
        ("subj07", (0, 35)),
        ("subj08", (0, 25)),
    ],
    "validation": [
        ("subj04", (0, 30)),
    ],
    "test": [
        ("subj05", (0, 30)),  # nb, subj05 still has 10 more sessions
    ],
    "testid": [
        ("subj01", (35, 40)),
        ("subj02", (35, 40)),
        ("subj03", (27, 32)),
        ("subj06", (27, 32)),
        ("subj07", (35, 40)),
        ("subj08", (25, 30)),
    ],
}

In [3]:
sub_ses_split_map = {}

for split in SUB_SES_SPLITS:
    for sub, (start, stop) in SUB_SES_SPLITS[split]:
        for ses_id in range(start, stop):
            sub_ses_split_map[sub, ses_id + 1] = split

In [4]:
targets = np.load(ROOT / "metadata/nsd_clip_coco_targets.npy")
print(targets.shape)

(73000,)


In [5]:
trial_df = pd.read_parquet(ROOT / "metadata/nsd_trial_metadata.parquet")

include_trial_ids = np.load(ROOT / "metadata/nsd_include_trial_ids.npy")
trial_df["include"] = trial_df.index.isin(include_trial_ids)

trial_df["split"] = [
    sub_ses_split_map.get((sub, ses)) for sub, ses in zip(trial_df["sub"], trial_df["ses"])
]

trial_df["target"] = targets[trial_df["nsd_id"]]

trial_df = trial_df.drop(columns="duration")

print(trial_df.shape)
trial_df.head(10)

(213000, 9)


,sub,ses,run,trial_id,onset,nsd_id,include,split,target
0,subj01,1,1,0,12.0,46002,False,train,19
1,subj01,1,1,1,16.0,61882,False,train,33
2,subj01,1,1,2,20.0,828,True,train,37
3,subj01,1,1,3,24.0,67573,False,train,37
4,subj01,1,1,4,28.0,16020,False,train,61
5,subj01,1,1,5,32.0,40422,True,train,22
6,subj01,1,1,6,36.0,51517,False,train,14
7,subj01,1,1,7,40.0,62325,False,train,8
8,subj01,1,1,8,44.0,50610,False,train,12
9,subj01,1,1,9,48.0,55065,True,train,59


In [6]:
trial_df.groupby("split").agg({"include": "sum"}).loc[["train", "validation", "test", "testid"]]

,include
split,
train,32539
validation,5418
test,5390
testid,5187


In [7]:
trial_df.to_csv(ROOT / "metadata/nsd_cococlip_metadata.csv", index=False)